# 04 · Analysis

Business questions answered off the star schema in `../warehouse.duckdb`.
Run `03_schema_build` first so the tables exist.

Questions:
1. How does ridership vary by month and weekday vs weekend?
2. Which zones/boroughs generate the most trips and revenue?
3. How do tipping behaviour and payment type relate?
4. What does the trip-distance / fare distribution look like?

In [ ]:
import duckdb, pandas as pd
import matplotlib.pyplot as plt
con = duckdb.connect('../warehouse.duckdb')
con.sql('SHOW TABLES').df()

## 1. Ridership by month + weekday/weekend

In [ ]:
by_month = con.sql("""
  SELECT d.month, d.month_name,
         COUNT(*) AS trips,
         ROUND(AVG(f.total_amount),2) AS avg_total,
         ROUND(SUM(f.total_amount),0) AS revenue
  FROM fact_trips f JOIN dim_date d USING (date_key)
  GROUP BY 1,2 ORDER BY 1
""").df()
by_month

In [ ]:
ax = by_month.plot(x='month_name', y='trips', kind='bar', legend=False, figsize=(9,3))
ax.set_title('Trips by month'); ax.set_ylabel('trips'); plt.tight_layout(); plt.show()

In [ ]:
con.sql("""
  SELECT CASE WHEN d.is_weekend THEN 'Weekend' ELSE 'Weekday' END AS day_type,
         COUNT(*) AS trips,
         ROUND(AVG(f.trip_distance),2) AS avg_miles,
         ROUND(AVG(f.total_amount),2)  AS avg_total
  FROM fact_trips f JOIN dim_date d USING (date_key)
  GROUP BY 1
""").df()

## 2. Top zones & boroughs by trips and revenue
Joins `dim_zone` on the **pickup** location.

In [ ]:
con.sql("""
  SELECT z.borough, z.zone,
         COUNT(*) AS trips,
         ROUND(SUM(f.total_amount),0) AS revenue
  FROM fact_trips f JOIN dim_zone z ON f.pu_location_id = z.location_id
  GROUP BY 1,2 ORDER BY trips DESC LIMIT 10
""").df()

In [ ]:
con.sql("""
  SELECT z.borough,
         COUNT(*) AS trips,
         ROUND(SUM(f.total_amount),0) AS revenue,
         ROUND(AVG(f.trip_distance),2) AS avg_miles
  FROM fact_trips f JOIN dim_zone z ON f.pu_location_id = z.location_id
  GROUP BY 1 ORDER BY trips DESC
""").df()

## 3. Tipping by payment type
Cash tips are typically unrecorded — expect near-zero tip rates on Cash vs Credit.

In [ ]:
con.sql("""
  SELECT p.payment_desc,
         COUNT(*) AS trips,
         ROUND(AVG(f.tip_amount),2) AS avg_tip,
         ROUND(100*AVG(CASE WHEN f.tip_amount>0 THEN 1 ELSE 0 END),1) AS pct_trips_tipped,
         ROUND(100*SUM(f.tip_amount)/NULLIF(SUM(f.fare_amount),0),1)  AS tip_pct_of_fare
  FROM fact_trips f JOIN dim_payment p USING (payment_type)
  GROUP BY 1 ORDER BY trips DESC
""").df()

## 4. Distance & fare distribution

In [ ]:
con.sql("""
  SELECT
    ROUND(quantile_cont(trip_distance, 0.50),2) AS median_miles,
    ROUND(quantile_cont(trip_distance, 0.90),2) AS p90_miles,
    ROUND(quantile_cont(total_amount, 0.50),2)  AS median_total,
    ROUND(quantile_cont(total_amount, 0.90),2)  AS p90_total,
    ROUND(AVG(duration_min),1)                  AS avg_duration_min
  FROM fact_trips
""").df()

In [ ]:
df = con.sql("SELECT trip_distance FROM fact_trips WHERE trip_distance <= 20").df()
df.hist(bins=40, figsize=(9,3)); plt.title('Trip distance (<=20 mi)'); plt.tight_layout(); plt.show()

## Takeaways
- Star schema makes each question a short, readable join — the payoff of the modeling in `03`.
- Expect Manhattan to dominate trips/revenue; Cash trips show ~0% recorded tips.
- To run on the **full year**, set the glob source in `02`/`03` and rebuild — every query here is unchanged.
- These same SELECTs run on Snowflake/BigQuery against external tables (see `docs/` productionizing notes).